In [112]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import difflib

load dataset

In [79]:
data=pd.read_csv("movies.csv")
data.shape

(4803, 24)

select only required columns for recommendation system

In [80]:
s_data=data[["genres","keywords","spoken_languages","overview","production_companies","production_countries","vote_average","cast","crew","director"]]
s_data.head()

,genres,keywords,spoken_languages,overview,production_companies,production_countries,vote_average,cast,crew,director
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...","In the 22nd century, a paraplegic Marine is di...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",7.2,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]","Captain Barbossa, long believed to be dead, ha...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",6.9,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,Action Adventure Crime,spy based on novel secret agent sequel mi6,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",A cryptic message from Bond’s past sends him o...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",6.3,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Following the death of District Attorney Harve...,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",7.6,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]","John Carter is a war-weary, former military ca...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",6.1,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


check no of null values

In [81]:
s_data.isnull().sum()

,0
genres,28
keywords,412
spoken_languages,0
overview,3
production_companies,0
production_countries,0
vote_average,0
cast,43
crew,0
director,30


fill null values with " " empty string

In [82]:
for columns_rows in s_data:
  if columns_rows=="vote_average":
    s_data[columns_rows]=s_data[columns_rows].fillna(0)
  s_data[columns_rows]=s_data[columns_rows].fillna('')

/tmp/ipykernel_3815/3398392697.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  s_data[columns_rows]=s_data[columns_rows].fillna('')
/tmp/ipykernel_3815/3398392697.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  s_data[columns_rows]=s_data[columns_rows].fillna(0)


In [83]:
s_data.head()

,genres,keywords,spoken_languages,overview,production_companies,production_countries,vote_average,cast,crew,director
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...","In the 22nd century, a paraplegic Marine is di...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",7.2,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]","Captain Barbossa, long believed to be dead, ha...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",6.9,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,Action Adventure Crime,spy based on novel secret agent sequel mi6,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",A cryptic message from Bond’s past sends him o...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",6.3,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Following the death of District Attorney Harve...,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",7.6,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]","John Carter is a war-weary, former military ca...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",6.1,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


combing all the selected features

In [84]:
combined_features=s_data["genres"]+' '+s_data["keywords"]+' '+s_data["spoken_languages"]+' '+s_data["overview"]+' '+s_data["production_companies"]+' '+s_data["production_countries"]+' '+s_data["cast"]+' '+s_data["crew"]+' '+s_data["director"]

In [85]:
combined_features.head()

,0
0,Action Adventure Fantasy Science Fiction cultu...
1,Adventure Fantasy Action ocean drug abuse exot...
2,Action Adventure Crime spy based on novel secr...
3,Action Crime Drama Thriller dc comics crime fi...
4,Action Adventure Science Fiction based on nove...


In [86]:
vec=TfidfVectorizer()

In [87]:
vec_features=vec.fit_transform(combined_features)

In [92]:
similarity=cosine_similarity(vec_features)
print(similarity)

[[1.         0.5757168  0.82522998 ... 0.32652876 0.08938812 0.15844721]
 [0.5757168  1.         0.58237051 ... 0.29610434 0.08231816 0.14519601]
 [0.82522998 0.58237051 1.         ... 0.3415958  0.09994837 0.16197072]
 ...
 [0.32652876 0.29610434 0.3415958  ... 1.         0.07300094 0.10598589]
 [0.08938812 0.08231816 0.09994837 ... 0.07300094 1.         0.04855406]
 [0.15844721 0.14519601 0.16197072 ... 0.10598589 0.04855406 1.        ]]


In [99]:
#get all movie titles so that we can find its index and recommend movies
titles=data["original_title"].tolist()
print(titles)

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz: The Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lant

In [100]:
movie_name=input("ENTER MOVIE NAME:")

ENTER MOVIE NAME:iron man


In [104]:
#the difflib is used to get the closest match for the given movie name with respect to the movie titles list.
closest=difflib.get_close_matches(movie_name,titles)
closest=closest[0]  #only top is enough
print(closest)

Iron Man


In [110]:
#get index of the closest movie
index=titles.index(closest)

In [116]:
#store each movie with its index and its similarity score with respect to closest movie
store=[]
for i in range(len(similarity[index])):
  store.append((i,similarity[index][i]))
print(store)

[(0, np.float64(0.46394634570831045)), (1, np.float64(0.3920136071622238)), (2, np.float64(0.48247348306677185)), (3, np.float64(0.4901414814638407)), (4, np.float64(0.47135681976868793)), (5, np.float64(0.45202263431719275)), (6, np.float64(0.4170153811440505)), (7, np.float64(0.50150399764799)), (8, np.float64(0.35899757587486913)), (9, np.float64(0.48897260002554815)), (10, np.float64(0.3567604982680603)), (11, np.float64(0.3138442450521952)), (12, np.float64(0.34872990686182115)), (13, np.float64(0.3720335516699805)), (14, np.float64(0.4374711825892962)), (15, np.float64(0.45700320386265325)), (16, np.float64(0.49503758979636137)), (17, np.float64(0.3942546833046541)), (18, np.float64(0.31204713039959886)), (19, np.float64(0.4741043154885631)), (20, np.float64(0.4819851033469222)), (21, np.float64(0.31517311022850675)), (22, np.float64(0.47746751623059114)), (23, np.float64(0.3895172005984983)), (24, np.float64(0.4013017973918249)), (25, np.float64(0.42393574430118286)), (26, np.fl

In [119]:
#sort the list in descending order with respect to similarity score
store=sorted(store,key=lambda x:x[1], reverse=True)


In [128]:
#print only top 20 movie match
for i in range(len(store)):
  if i>19:
    break
  index=store[i][0]
  print(i+1,titles[index])


1 Iron Man
2 Iron Man 2
3 Ant-Man
4 Hulk
5 Captain America: Civil War
6 Blade: Trinity
7 Spider-Man
8 Spider-Man 2
9 Avengers: Age of Ultron
10 Terminator 3: Rise of the Machines
11 Alexander
12 Iron Man 3
13 Catwoman
14 Elektra
15 The Core
16 Blade II
17 The Day After Tomorrow
18 The Punisher
19 Batman Begins
20 The Bourne Ultimatum


Full work

In [129]:
#use all the parts into one for easy understand
movie_name=input("ENTER MOVIE NAME:")
closest=difflib.get_close_matches(movie_name,titles)
closest=closest[0]  #only top is enough
index=titles.index(closest)
store=[]
for i in range(len(similarity[index])):
  store.append((i,similarity[index][i]))
store=sorted(store,key=lambda x:x[1], reverse=True)
for i in range(len(store)):
  if i>19:
    break
  index=store[i][0]
  print(i+1,titles[index])

ENTER MOVIE NAME:king kong
1 King Kong
2 The Hobbit: The Desolation of Smaug
3 The Hobbit: The Battle of the Five Armies
4 War of the Worlds
5 The Chronicles of Riddick
6 Alexander
7 The Day After Tomorrow
8 The Core
9 Jurassic World
10 Master and Commander: The Far Side of the World
11 The Dark Knight Rises
12 Ant-Man
13 Catwoman
14 Kill Bill: Vol. 1
15 Harry Potter and the Chamber of Secrets
16 Contact
17 Minority Report
18 Harry Potter and the Prisoner of Azkaban
19 A.I. Artificial Intelligence
20 Harry Potter and the Philosopher's Stone
